In [1]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import numpy as np
from shapely.geometry import mapping
from shapely import geometry
from scipy import ndimage
from shapely import wkb
from xgboost import XGBClassifier
import json
import sys
# append the path of the parent directory
sys.path.append("..")
from utils.dol import *

def binary_classification_metrics(
    df: pd.DataFrame,
    target_column: str,
    pred_column: str
):
    """
    Compute accuracy, precision (class 1), and recall (class 1)
    from a dataframe containing binary labels (0/1).

    Returns a dict with metrics.
    """

    y_true = df[target_column].astype(int)
    y_pred = df[pred_column].astype(int)

    # Confusion matrix components
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()

    # Metrics
    accuracy = (tp + tn) / len(df) if len(df) > 0 else 0.0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "precision_class_1": precision_1,
        "recall_class_1": recall_1,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

CACHE_DIR = "/app/data/datasets/debug/bush/cache"
MODEL_SAVE_FOLDER = "/app/data/datasets/debug/bush/models"
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_FOLDER,'dol_xgb_model_v1.1.json')
METADATA_SAVE_PATH = os.path.join(MODEL_SAVE_FOLDER,'dol_xgb_model_v1.1_metadata.json')
load_dotenv()

db_string = os.getenv('DB_STRING_PROD')
engine = create_engine(db_string)

In [2]:
gdf_forests_zones = gpd.read_file('/app/data/datasets/debug/bush/sources/ocsge_forests_clean_types_v3.gpkg',driver='GPKG')
gdf_waters_zones = gpd.read_file('/app/data/datasets/debug/bush/sources/COURS_D_EAU.shp')
gdf_u_zone = gpd.read_file('/app/data/datasets/debug/bush/sources/34_zones_u.gpkg',driver='GPKG')
gdf_forests_zones = gdf_forests_zones[gdf_forests_zones.forest_type==1]

/opt/conda/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
/opt/conda/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


In [3]:
gdf_zones_r = gpd.read_postgis("select * from detections.n_dfci_old50m_s_034_ilots", geom_col='geom',con=engine)
gdf_zones_r

,id_ilot,code_ilot,compte_communal,insee_com,geom
0,103918,340198V00297_0,340198V00297,34198,"MULTIPOLYGON (((777980.658 6273167.472, 777960..."
1,104534,340199L00953_0,340199L00953,34199,"MULTIPOLYGON (((734417.82 6262732.91, 734410.0..."
2,104606,340199M01453_0,340199M01453,34199,"MULTIPOLYGON (((733334.706 6262686.729, 733326..."
3,104637,340199N00167_0,340199N00167,34199,"MULTIPOLYGON (((733716.05 6262894.98, 733716.4..."
4,104738,340199S00687_0,340199S00687,34199,"MULTIPOLYGON (((734263.5 6262804.4, 734265.23 ..."
...,...,...,...,...,...
168534,105967,340202D00714_0,340202D00714,34202,"MULTIPOLYGON (((760870.94 6276603.97, 760873.0..."
168535,106032,340202F00387_0,340202F00387,34202,"MULTIPOLYGON (((761060.53 6276468.42, 761062.0..."
168536,7323,340015N00005_0,340015N00005,34015,"MULTIPOLYGON (((690956.04 6255317.9, 690953.39..."
168537,2195,340006C00083_0,340006C00083,34006,"MULTIPOLYGON (((683512.99 6248082.81, 683520.8..."


In [4]:
# reload model
xgb_model_reloaded = XGBClassifier()
xgb_model_reloaded.load_model(MODEL_SAVE_PATH)

# reload metadata
with open(METADATA_SAVE_PATH) as f:
    metadata = json.load(f)
xgb_model_reloaded.scale_pos_weight = metadata["scale_pos_weight"]
THRESHOLD = metadata["decision_threshold"]
metadata

{'decision_threshold': 0.36363636363636365,
 'metric_optimized': 'f1',
 'scale_pos_weight': 1.0885245901639344}

# 1. Analysis on testset : Boissière

In [5]:
# load : 
df_test = pd.read_parquet(os.path.join(CACHE_DIR,'test.parquet'))
y = df_test['target_local_control']
x = df_test.drop(columns=['target_local_control','sample_id'])

In [6]:
x_test_business_features = x[['has_contact_river_zone','has_contact_forest_zone','has_contact_u_zone','has_inhabited_building','has_building']]
x_test_ml_features = x.drop(columns=['has_contact_river_zone','has_contact_forest_zone','has_contact_u_zone'])

In [7]:
gdf_datas = gpd.read_file('/app/data/datasets/debug/bush/labels/target_dol_zones_v1.gpkg',driver='GPKG')

gdf_datas.to_crs('EPSG:2154', inplace=True)
gdf_zone_datas = gdf_datas[gdf_datas.insee_com=='34035']
gdf_zone_datas.head(5)

/opt/conda/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


,image_path,id_ilot,code_ilot,compte_communal,insee_com,target_control,target_pv,geometry
878,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19730,340035+00001_0,340035+00001,34035,0.0,0.0,"MULTIPOLYGON (((752023.38 6285086.06, 752021.8..."
879,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19731,340035+00003_0,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751926.52 6285261.73, 751928.5..."
880,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19733,340035+00003_2,340035+00003,34035,1.0,1.0,"MULTIPOLYGON (((751889.05 6284995.31, 751908.5..."
881,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19734,340035+00003_3,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751407.76 6282780.51, 751405.1..."
882,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19735,340035+00003_4,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751515.53 6282876.27, 751514.0..."


In [8]:
y_proba = xgb_model_reloaded.predict_proba(x_test_ml_features)[:, 1]
 
gdf_zone_datas["proba_control"] = y_proba
gdf_zone_datas["pred_control"] = 0
gdf_zone_datas.loc[gdf_zone_datas["proba_control"] >= THRESHOLD,"pred_control"] = 1
gdf_zone_datas

/opt/conda/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/opt/conda/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,image_path,id_ilot,code_ilot,compte_communal,insee_com,target_control,target_pv,geometry,proba_control,pred_control
878,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19730,340035+00001_0,340035+00001,34035,0.0,0.0,"MULTIPOLYGON (((752023.38 6285086.06, 752021.8...",0.043421,0
879,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19731,340035+00003_0,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751926.52 6285261.73, 751928.5...",0.061792,0
880,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19733,340035+00003_2,340035+00003,34035,1.0,1.0,"MULTIPOLYGON (((751889.05 6284995.31, 751908.5...",0.756004,1
881,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19734,340035+00003_3,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751407.76 6282780.51, 751405.1...",0.043150,0
882,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,19735,340035+00003_4,340035+00003,34035,0.0,0.0,"MULTIPOLYGON (((751515.53 6282876.27, 751514.0...",0.766932,1
...,...,...,...,...,...,...,...,...,...,...
1414,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,20283,340035W00009_0,340035W00009,34035,1.0,1.0,"MULTIPOLYGON (((752968.29 6283179.48, 752980.8...",0.914036,1
1415,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,20284,340035Z00001_0,340035Z00001,34035,1.0,1.0,"MULTIPOLYGON (((751619.601 6284418.833, 751615...",0.917902,1
1416,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,20285,340035Z00002_0,340035Z00002,34035,0.0,0.0,"MULTIPOLYGON (((752284.406 6285301.143, 752279...",0.138814,0
1417,/app/runs/aigle_aerial_yolov_2024_boissiere_34...,20286,340035Z00003_0,340035Z00003,34035,0.0,0.0,"MULTIPOLYGON (((751395.27 6283379.04, 751406 6...",0.451280,1


In [9]:
df_features_imp = pd.DataFrame(columns = ['features', 'importance'], data = np.asarray([x_test_ml_features.columns, xgb_model_reloaded.feature_importances_]).T)
df_features_imp.sort_values(by='importance',ascending=False).head(20)

,features,importance
6,z50_12_mean,0.34097
63,surf_area,0.063309
48,z30_12_count_gt_100_ratio,0.034271
39,z30_12_nb_surf_sup1m2_200,0.027468
3,z30_12_mean,0.024756
51,z50_12_count_gt_100_ratio,0.022048
53,z50_14_count_gt_100_ratio,0.021862
34,z50_13_nb_surf_sup1m2_100,0.019666
64,has_building,0.019414
32,z30_14_nb_surf_sup1m2_100,0.018052


In [10]:
test_results_gdf = postprocess_pred_control(gdf_zone_datas, x_test_business_features)

In [11]:
binary_classification_metrics(test_results_gdf,'target_control','pred_control_pp')

{'accuracy': np.float64(0.7874306839186691),
 'precision_class_1': np.float64(0.859375),
 'recall_class_1': np.float64(0.6521739130434783),
 'tp': 165,
 'fp': 27,
 'fn': 88,
 'tn': 261}

In [12]:
binary_classification_metrics(test_results_gdf,'target_pv','pred_control_pp')

{'accuracy': np.float64(0.7837338262476895),
 'precision_class_1': np.float64(0.7916666666666666),
 'recall_class_1': np.float64(0.6637554585152838),
 'tp': 152,
 'fp': 40,
 'fn': 77,
 'tn': 272}

In [13]:
test_results_gdf.to_file(
    "/app/data/datasets/debug/bush/pred_boissiere_34035_dol_zones_xgb_v1.1.gpkg",
    driver="GPKG"
)

# 2. analysis on unknown geozone : Puisserguier

In [14]:
communes_test = [
    {'name':'puisserguier', 'geozone_code': 34225,'geozone_id': 297,'result_segmentation_files':  ['/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_69.tif',
        '/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_70.tif','/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_83.tif',
        '/app/runs/aigle_aerial_yolov_2024_puisserguier_34225_v1.1/results/test_zonal_AERIAL_LABEL-COSIA_class-prob_84.tif']}
]
imgs_bounds = []
for comm in communes_test :
    for img_path in comm['result_segmentation_files']:
        with rasterio.open(img_path) as src :
            bbox = src.bounds
            bbox_polygon = geometry.box(*bbox)
            print(bbox_polygon)
            imgs_bounds.append([img_path, bbox_polygon])

gdf_img = gpd.GeoDataFrame(data= imgs_bounds, columns=['image_path','geometry'], geometry='geometry', crs='EPSG:2154')
gdf_img.to_crs('EPSG:2154',inplace=True)
gdf_img.drop_duplicates(subset='image_path',inplace=True)

POLYGON ((705000 6250000, 705000 6255000, 700000 6255000, 700000 6250000, 705000 6250000))
POLYGON ((705000 6255000, 705000 6260000, 700000 6260000, 700000 6255000, 705000 6255000))
POLYGON ((710000 6250000, 710000 6255000, 705000 6255000, 705000 6250000, 710000 6250000))
POLYGON ((710000 6255000, 710000 6260000, 705000 6260000, 705000 6255000, 710000 6255000))


In [15]:
gdf_zone_data = gdf_zones_r[gdf_zones_r.insee_com.isin([str(x['geozone_code']) for x in communes_test])]
gdf_zone_data


,id_ilot,code_ilot,compte_communal,insee_com,geom
729,114460,340225A00358_0,340225A00358,34225,"MULTIPOLYGON (((700437.59 6254693.26, 700432.1..."
730,114567,340225G00055_0,340225G00055,34225,"MULTIPOLYGON (((703080.01 6252499.288, 703082...."
14674,114611,340225K00023_0,340225K00023,34225,"MULTIPOLYGON (((703635.706 6252494.598, 703631..."
20255,114542,340225D00225_1,340225D00225,34225,"MULTIPOLYGON (((702967.56 6252591.13, 702985.0..."
20256,114552,340225E00124_0,340225E00124,34225,"MULTIPOLYGON (((703010.577 6252719.254, 703002..."
...,...,...,...,...,...
117146,114712,340225V00242_0,340225V00242,34225,"MULTIPOLYGON (((701822.369 6253245.245, 701825..."
117147,114713,340225V00258_0,340225V00258,34225,"MULTIPOLYGON (((700596.25 6254784.28, 700601.6..."
117148,114715,340225V00285_0,340225V00285,34225,"MULTIPOLYGON (((704360.6 6253567.9, 704365.651..."
117150,114716,340225Y00008_0,340225Y00008,34225,"MULTIPOLYGON (((703313.423 6254541.882, 703308..."


In [16]:
gdf_zone_data = gpd.sjoin(gdf_img,gdf_zone_data, how='right', predicate='intersects').drop(columns='index_left')
gdf_zone_data = gdf_zone_data[~gdf_zone_data.image_path.isna()]
gdf_zone_data.rename(columns={'geom':'geometry'}, inplace=True)
#gdf_zone_data.set_geometry("geometry",inplace=True)
gdf_zone_data

,image_path,id_ilot,code_ilot,compte_communal,insee_com,geometry
729,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114460,340225A00358_0,340225A00358,34225,"MULTIPOLYGON (((700437.59 6254693.26, 700432.1..."
730,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114567,340225G00055_0,340225G00055,34225,"MULTIPOLYGON (((703080.01 6252499.288, 703082...."
14674,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114611,340225K00023_0,340225K00023,34225,"MULTIPOLYGON (((703635.706 6252494.598, 703631..."
20255,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114542,340225D00225_1,340225D00225,34225,"MULTIPOLYGON (((702967.56 6252591.13, 702985.0..."
20256,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114552,340225E00124_0,340225E00124,34225,"MULTIPOLYGON (((703010.577 6252719.254, 703002..."
...,...,...,...,...,...,...
117146,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114712,340225V00242_0,340225V00242,34225,"MULTIPOLYGON (((701822.369 6253245.245, 701825..."
117147,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114713,340225V00258_0,340225V00258,34225,"MULTIPOLYGON (((700596.25 6254784.28, 700601.6..."
117148,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114715,340225V00285_0,340225V00285,34225,"MULTIPOLYGON (((704360.6 6253567.9, 704365.651..."
117150,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114716,340225Y00008_0,340225Y00008,34225,"MULTIPOLYGON (((703313.423 6254541.882, 703308..."


In [17]:
x_ml_features, x_business_features =  preprocess_features(gdf_zone_data, gdf_forests_zones, gdf_waters_zones, gdf_u_zone, cache_dir = CACHE_DIR, debug=False)
x_ml_features

,z10_12_mean,z10_13_mean,z10_14_mean,z30_12_mean,z30_13_mean,z30_14_mean,z50_12_mean,z50_13_mean,z50_14_mean,z10_12_surf_gt_100_ratio,...,z10_14_count_gt_200_ratio,z30_12_count_gt_200_ratio,z30_13_count_gt_200_ratio,z30_14_count_gt_200_ratio,z50_12_count_gt_200_ratio,z50_13_count_gt_200_ratio,z50_14_count_gt_200_ratio,surf_area,has_building,has_inhabited_building
0,0.081943,0.187314,0.006430,0.052628,0.090993,0.001868,0.000000,0.000000,0.000000,0.035785,...,0.000011,0.013349,0.059187,0.000000,0.000000,0.000000,0.000000,2838.593550,1,1
0,0.023775,0.002703,0.025163,0.000042,0.000004,0.000037,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,200.710186,0,0
0,0.000375,0.000000,0.000251,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,27.558490,0,0
0,0.132534,0.021827,0.022008,0.041063,0.009847,0.014145,0.000000,0.000000,0.000000,0.125909,...,0.000000,0.010014,0.000000,0.000000,0.000000,0.000000,0.000000,1024.497700,1,1
0,0.000549,0.000000,0.004873,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,36.130231,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,0.084710,0.235366,0.003627,0.049886,0.296383,0.007542,0.021001,0.178285,0.015281,0.036680,...,0.000000,0.029171,0.260763,0.000330,0.006902,0.139177,0.001968,12556.009081,1,1
0,0.017066,0.006161,0.013689,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.022148,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,201.558650,1,1
0,0.094484,0.001696,0.029076,0.037224,0.000621,0.013179,0.011668,0.000147,0.005945,0.053226,...,0.000000,0.017990,0.000000,0.000000,0.001507,0.000000,0.000000,5846.863459,1,1
0,0.034198,0.089547,0.202880,0.008229,0.028856,0.136187,0.001961,0.034827,0.030680,0.000000,...,0.012549,0.000000,0.003802,0.090286,0.000000,0.020847,0.006774,8048.419669,0,0


In [18]:
y_proba = xgb_model_reloaded.predict_proba(x_ml_features)[:, 1]

#y_pred_custom = (y_proba >= THRESHOLD)
gdf_zone_data["proba_control"] = y_proba
gdf_zone_data["pred_control"] = 0
gdf_zone_data.loc[gdf_zone_data["proba_control"] >= THRESHOLD,"pred_control"] = 1
gdf_zone_data


,image_path,id_ilot,code_ilot,compte_communal,insee_com,geometry,proba_control,pred_control
729,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114460,340225A00358_0,340225A00358,34225,"MULTIPOLYGON (((700437.59 6254693.26, 700432.1...",0.729936,1
730,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114567,340225G00055_0,340225G00055,34225,"MULTIPOLYGON (((703080.01 6252499.288, 703082....",0.043148,0
14674,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114611,340225K00023_0,340225K00023,34225,"MULTIPOLYGON (((703635.706 6252494.598, 703631...",0.043148,0
20255,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114542,340225D00225_1,340225D00225,34225,"MULTIPOLYGON (((702967.56 6252591.13, 702985.0...",0.496302,1
20256,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114552,340225E00124_0,340225E00124,34225,"MULTIPOLYGON (((703010.577 6252719.254, 703002...",0.043148,0
...,...,...,...,...,...,...,...,...
117146,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114712,340225V00242_0,340225V00242,34225,"MULTIPOLYGON (((701822.369 6253245.245, 701825...",0.641022,1
117147,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114713,340225V00258_0,340225V00258,34225,"MULTIPOLYGON (((700596.25 6254784.28, 700601.6...",0.044824,0
117148,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114715,340225V00285_0,340225V00285,34225,"MULTIPOLYGON (((704360.6 6253567.9, 704365.651...",0.237024,0
117150,/app/runs/aigle_aerial_yolov_2024_puisserguier...,114716,340225Y00008_0,340225Y00008,34225,"MULTIPOLYGON (((703313.423 6254541.882, 703308...",0.644411,1


In [19]:
gdf_zone_data.set_geometry("geometry",inplace=True)

In [20]:
test_results_gdf = postprocess_pred_control(gdf_zone_data, x_business_features)

test_results_gdf.to_file(
    "/app/data/datasets/debug/bush/pred_puisserguier_34225_dol_zones_xgb_v1.1.gpkg",
    driver="GPKG"
)